# Hand-Pose Confidence Estimation under Occlusion — GPU run

Runs the real experiment: **HaMeR** on **HO-3D v3**, four uncertainty estimators,
occlusion-stratified analysis. Pipeline already validated end-to-end on synthetic
data (`scripts/synthetic_validation.py`).

**Before running — must exist in `MyDrive/HO3D_v3/`:**
- `HO3D_pilot.zip` (ABF10 + MDF10 + SM2, 5,087 frames)
- `HO3D_v3_segmentations_rendered.zip`
- `MANO_RIGHT.pkl` (chumpy-free version)

Runtime → Change runtime type → **T4 GPU**. Cells are idempotent; on a fresh
runtime run top to bottom (~15 min of setup).

In [ ]:
# 0. GPU + Drive (mount is idempotent — safe to re-run)
!nvidia-smi -L
import os
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
print(os.listdir('/content/drive/MyDrive/HO3D_v3'))

In [ ]:
# 1. Get the project code
!git clone https://github.com/katherinesiyuchen-sys/hamer-confidence /content/proj
%cd /content/proj
!pip install -r requirements.txt

In [ ]:
# 2. Install HaMeR core only. --no-deps skips mmcv/detectron2/chumpy, which are
#    demo-only deps that can't build on py3.13 — crops come from HO-3D GT instead.
#    ViTPose is not installed at all.
%cd /content
!git clone --recursive https://github.com/geopavlakos/hamer.git
%cd /content/hamer
!pip install -e . --no-deps
!pip install pytorch-lightning "smplx==0.1.28" yacs timm einops scikit-image pyrender hydra-core pyrootutils rich webdataset

# checkpoints (~6GB): use Drive cache if present, else fetch once and cache
import os
if os.path.isdir('/content/drive/MyDrive/hamer_DATA'):
    !cp -r /content/drive/MyDrive/hamer_DATA /content/hamer/_DATA
else:
    !bash fetch_demo_data.sh
    !cp -r /content/hamer/_DATA /content/drive/MyDrive/hamer_DATA

# MANO hand model (license-gated, lives on Drive) + path link
!mkdir -p /content/hamer/_DATA/data/mano
!cp /content/drive/MyDrive/HO3D_v3/MANO_RIGHT.pkl /content/hamer/_DATA/data/mano/
%cd /content/proj
!ln -sf /content/hamer/_DATA /content/proj/_DATA

In [ ]:
# 3. Smoke test: HaMeR loads and runs
import sys; sys.path.insert(0, '/content/proj')
from src.hamer_wrapper import HamerPredictor
import numpy as np, cv2
pred = HamerPredictor()
print('HaMeR loaded OK')

In [ ]:
%%shell
unzip -q -n /content/drive/MyDrive/HO3D_v3/HO3D_pilot.zip -d /content/HO3D
unzip -q -n /content/drive/MyDrive/HO3D_v3/HO3D_v3_segmentations_rendered.zip -d /content/HO3D_seg || true
ls /content/HO3D/train/

In [ ]:
# 4b. Dataset sanity check
HO3D = '/content/HO3D'
from src.ho3d_data import HO3D as HO3DDataset
ds = HO3DDataset(HO3D, split='train')
print(f'{len(ds)} frames available')   # expect 5087
fr = ds[0]
print('first frame:', fr.seq, fr.idx, fr.image_path.exists())

In [ ]:
# 5. PILOT: 500 frames end-to-end (~20-40 min on T4)
!python scripts/run_eval.py --ho3d $HO3D --out /content/drive/MyDrive/results_pilot \
    --stage all --max-frames 500 --tta 4

In [ ]:
# 6. Check per_occlusion_bin is non-empty before committing to the full run
import json
print(json.dumps(json.load(open('/content/drive/MyDrive/results_pilot/report.json')), indent=2))

In [ ]:
# 7. FULL RUN (resumable; results cache to Drive)
!python scripts/run_eval.py --ho3d $HO3D --out /content/drive/MyDrive/results_full \
    --stage all --max-frames 20000 --tta 8

In [ ]:
# 8. Figures
from src import plots
plots.make_all('/content/drive/MyDrive/results_full/scores.npz',
               '/content/drive/MyDrive/results_full/figures')
from IPython.display import Image, display
for f in ['sparsification','occlusion_stratified','filtering','score_vs_error']:
    display(Image(f'/content/drive/MyDrive/results_full/figures/{f}.png'))

## Notes
- **Occlusion masks**: `run_eval.py` reads per-frame masks from `results*/masks/`.
  Render them with `src/ho3d_data.amodal_object_mask` (needs YCB object meshes from the
  HO-3D README) — without them, everything still runs but occlusion stratification is skipped.
- **Resumable**: inference caches per-frame `.npz`; re-running skips finished frames.
- **Colab disconnects**: results live on Drive, so nothing is lost.
